In [0]:
import requests
import pandas as pd

files = [
    "map_cities",
    "map_cancellation_reasons",
    "bulk_rides",
    "map_payment_methods",
    "map_ride_statuses",
    "map_vehicle_makes",
    "map_vehicle_types"
]

sas_token = "sp=rl&st=2026-08-15T09:16:42Z&se=2026-08-17T17:31:42Z&spr=https&sv=2026-02-06&sr=c&sig=DXsbXTErDt0XIN9775xBes7%2FmIU7yJjvvLqhqRXuar0%3D"

base_url = "https://projectubersa.blob.core.windows.net/raw"

for file_name in files:

    url = f"{base_url}/ingestion/{file_name}.json?{sas_token}"

    response = requests.get(url)

    print(f"{file_name} -> {response.status_code}")

    if response.status_code != 200:
        print(response.text[:500])
        continue

    try:
        data = response.json()

        df = pd.DataFrame(data)

        print(f"Rows: {len(df)}")
        print(f"Columns: {df.columns.tolist()}")

        df_spark = spark.createDataFrame(df)

        df_spark.write \
            .format("delta") \
            .mode("overwrite") \
            .opting("overwriteSchema", True) \
            .saveAsTable(f"uber.bronze.{file_name}")

        print(f"✓ {file_name} loaded into uber.bronze.{file_name}")

    except Exception as e:
        print(f"✗ Error processing {file_name}: {e}")

In [0]:
%sql
select * from uber.bronze.rides_raw